# b_Build_Model Runner

각 피험자 `Model_osim/` 폴더에 들어온 AddBiomech 산출물을 다음 파이프라인으로 처리:

1. **rename_AddBiomech_model** — `match_markers_but_ignore_physics.osim` → `SUB{num}_Scaled.osim`
2. **add_hand_mass_model**     — `SUB{num}_Scaled_HeavyHand_{w}kg.osim` 생성 (각 손에 w/2 kg)
3. **add_box_weldjoint_model** — `SUB{num}_Scaled_{WeldBox|SplitBox}_{w}kg.osim` 생성 (ADDBOX 활용)

각 단계의 osim 이 만들어진 직후, 같은 셀에서 **add_reserve_actuators** 가
해당 파일에 reserve / residual / torque `CoordinateActuator` 를 in-place 주입한다
(파일명 변경 없음, 이미 있으면 skip).

피험자별 가능한 박스 무게(w_kg)는 `SUB_Info.subjects[namecode]['conditions']` 키의 prefix
(`7kg`, `10kg`, `15kg`)에서 자동 추출됩니다.

In [1]:
import os
import sys

repo_root = os.getcwd()
if not os.path.isdir(os.path.join(repo_root, 'Codes')):  # 노트북 실행 위치가 repo root 가 아닌 경우
    repo_root = r'C:/Users/ok/Documents/GitHub/BOX'      # (optional)TODO: change to your path

work_dir = os.path.join(repo_root, 'Codes', 'b_Build_Model')
codes_dir = os.path.dirname(work_dir)
os.chdir(work_dir)
for p in (work_dir, codes_dir):
    if p not in sys.path:
        sys.path.insert(0, p)

print('cwd      :', os.getcwd())
print('work_dir :', work_dir)

cwd      : C:\Users\ok\Documents\GitHub\BOX\Codes\b_Build_Model
work_dir : C:/Users/ok/Documents/GitHub/BOX\Codes\b_Build_Model


In [2]:
from SUB_Info import subjects

print('Available subjects:', list(subjects.keys()))

namecode = '260512_KCH'  # TODO: 필요 시 변경
info = subjects[namecode]
print('sex       :', info['sex'])
print('protocol  :', info['protocol'])
print('conditions:', list(info['conditions'].keys()))

Available subjects: ['240124_PJH', '260306_KTY', '260423_CES', '260512_HSH', '260512_KCH', '260519_SHY', '260519_KMJ', '260521_JSY', '260521_KJA', '260526_PJH', '260526_PJM']
sex       : M
protocol  : Asymmetric
conditions: ['7kg_10bpm', '15kg_10bpm', '7kg_16bpm', '15kg_16bpm']


## 1) rename_AddBiomech_model

`match_markers_but_ignore_physics.osim` → `SUB{num}_Scaled.osim`

이어서 `add_reserve_actuators` 로 동일 파일에 CoordinateActuator 주입.

In [ ]:
from rename_AddBiomech_model import rename_scaled_model
from add_reserve_actuators import add_reserve_actuators

scaled_path = rename_scaled_model(namecode, overwrite=False, add_actuators=False)
if scaled_path:
    add_reserve_actuators(scaled_path)
print('Scaled model:', scaled_path)

## 2) add_hand_mass_model

7kg → 각 손 +3.5kg, 10kg → +5kg, 15kg → +7.5kg

생성(또는 기존) 각 HeavyHand osim 에 이어서 `add_reserve_actuators` 주입.

In [ ]:
from add_hand_mass_model import build_heavyhand_models
from add_reserve_actuators import add_reserve_actuators

heavyhand_outputs = build_heavyhand_models(
    namecode, overwrite=True, add_actuators=False)
for p in heavyhand_outputs:
    add_reserve_actuators(p)
    print(' -', p)

## 3) add_box_weldjoint_model

`ADDBOXtoOSIM(..., Constraint=True)`  → `WeldBox`
`ADDBOXtoOSIM(..., Constraint=False)` → `SplitBox`

생성(또는 기존) 각 WeldBox / SplitBox osim 에 이어서 `add_reserve_actuators` 주입.

In [ ]:
from add_box_weldjoint_model import build_box_models
from add_reserve_actuators import add_reserve_actuators

box_outputs = build_box_models(
    namecode, overwrite=True, add_actuators=False)
for p in box_outputs:
    add_reserve_actuators(p)
    print(' -', p)

## (옵션) 전체 피험자 일괄 실행

In [ ]:
from rename_AddBiomech_model import rename_all
from add_hand_mass_model import build_all as build_all_heavyhand
from add_box_weldjoint_model import build_all as build_all_box
from add_reserve_actuators import add_reserve_actuators

for path in rename_all(overwrite=False, add_actuators=False).values():
    if path:
        add_reserve_actuators(path)

for paths in build_all_heavyhand(overwrite=True, add_actuators=False).values():
    for p in paths:
        add_reserve_actuators(p)

for paths in build_all_box(overwrite=True, add_actuators=False).values():
    for p in paths:
        add_reserve_actuators(p)